# SWU Card Scanner - train the card detector

Trains a **YOLO11s-pose** model that finds Star Wars: Unlimited cards in a photo and regresses their four corners, then exports it to ONNX for the browser app.

One class (`card`), four keypoints (TL, TR, BR, BL in the card's own frame). The model is deliberately card-agnostic - it never learns *which* card it is looking at, so new sets never require retraining.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** (or better).

Everything is regenerated on the Colab machine, so nothing large needs uploading: only the three Python scripts from `SWUCardScanner/model/`.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> T4 GPU'

## 2. Install dependencies

In [ ]:
%pip install -q ultralytics onnx onnxslim onnxruntime opencv-python-headless requests tqdm
import ultralytics; ultralytics.checks()

## 3. Get the scripts onto the machine

Run **one** of the two cells below.

- **Upload** is the quickest: pick `scrape_cards.py` and `build_dataset.py` from `SWUCardScanner/model/`.
- **Drive** is better if you retrain often: copy the `model` folder to `MyDrive/SWUCardScanner/model` once and just remount each session.

In [ ]:
# --- Option A: upload scrape_cards.py and build_dataset.py from your machine ---
import os
os.makedirs('/content/model', exist_ok=True)
%cd /content/model
from google.colab import files
files.upload()  # select scrape_cards.py AND build_dataset.py
print(sorted(os.listdir('/content/model')))

In [ ]:
# --- Option B: pull them from Google Drive ---
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/model && cp /content/drive/MyDrive/SWUCardScanner/model/*.py /content/model/
# %cd /content/model
# import os; print(sorted(os.listdir('/content/model')))

## 4. Download the card art

Pulls every unique card image from `api.sw-unlimited-db.com` (~3k PNGs at 440px, a few minutes on Colab's connection). Re-run any time a new set drops - **no retraining needed for new cards**, this only matters because more art means more visual variety in the training scenes.

In [ ]:
!python scrape_cards.py --width 440 --workers 16

## 5. (Optional but recommended) Real background photos

The generator ships with procedural backgrounds, but nothing beats 20-50 photos of *your* table, playmat, desk and carpet under the lighting you actually scan in. Drop them in `/content/model/backgrounds` and they get used for ~55% of scenes.

Skip this cell to train on procedural backgrounds only.

In [ ]:
import os
os.makedirs('/content/model/backgrounds', exist_ok=True)
%cd /content/model/backgrounds
from google.colab import files
files.upload()  # select your background photos (jpg/png)
%cd /content/model
print(len(os.listdir('/content/model/backgrounds')), 'background photos')

## 6. Build the synthetic dataset

8000 scenes takes roughly 10-15 minutes across Colab's CPU cores. Bump `--scenes` if the model underfits; 4000 is enough for a first look.

In [ ]:
!python build_dataset.py --scenes 8000 --size 768 --workers 0
!cat dataset/data.yaml

### Eyeball the labels before burning GPU hours on them

In [ ]:
import glob, random, cv2, numpy as np
import matplotlib.pyplot as plt

COLOURS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]  # TL TR BR BL
picks = random.sample(sorted(glob.glob('dataset/images/train/*.jpg')), 6)
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax, path in zip(axes.ravel(), picks):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    label = path.replace('/images/', '/labels/').replace('.jpg', '.txt')
    for line in open(label):
        v = [float(x) for x in line.split()[1:]]
        if len(v) != 12:
            continue
        pts = (np.array(v[4:], np.float32).reshape(4, 2) * [w, h]).astype(int)
        cv2.polylines(img, [pts], True, (255, 0, 255), 2)
        for pt, c in zip(pts, COLOURS):
            cv2.circle(img, tuple(pt), 7, c, -1)
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Train

Notes on the settings:

- `fliplr=0, flipud=0` - **must stay off.** Flipping would swap the corner ordering and destroy the orientation signal the un-warp step depends on.
- `pose=25.0` weights corner accuracy well above box accuracy; a loose box is fine, a sloppy corner is not.
- `degrees=3` only, because the dataset already contains full 360-degree rotation baked into the scenes.
- Drop `batch` to 8 if the T4 runs out of memory at 768px.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s-pose.pt')
results = model.train(
    data='dataset/data.yaml',
    epochs=80,
    patience=25,
    imgsz=768,
    batch=16,          # 16 fits a T4 at 768px; use 32+ on an A100
    cos_lr=True,
    close_mosaic=12,
    box=7.5,
    pose=25.0,
    kobj=1.0,
    cls=0.3,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.5,
    degrees=3.0, translate=0.1, scale=0.4, shear=2.0,
    perspective=0.0005,
    fliplr=0.0, flipud=0.0,   # keep corner ordering meaningful
    mosaic=1.0, mixup=0.15,
    project='runs', name='swu-detect', exist_ok=True,
)

In [ ]:
from IPython.display import Image, display
display(Image('runs/swu-detect/results.png'))
display(Image('runs/swu-detect/val_batch0_pred.jpg'))

## 8. Export to ONNX for the browser

`opset=12` and `simplify=True` keep onnxruntime-web happy. `dynamic=False` bakes in the 768x768 input, which is what the web app letterboxes to.

In [ ]:
from ultralytics import YOLO
best = YOLO('runs/swu-detect/weights/best.pt')
onnx_path = best.export(format='onnx', imgsz=768, opset=12, simplify=True, dynamic=False)
print('exported:', onnx_path)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
for i in sess.get_inputs():
    print('input ', i.name, i.shape, i.type)
for o in sess.get_outputs():
    print('output', o.name, o.shape, o.type)
print('\nExpect output [1, 13, 12096]: 4 box + 1 class + 4 corners x 2 coords.')

### Optional: int8 quantised build

Roughly 4x smaller and noticeably faster on phones, at some cost in corner precision. Ship both and let the app switch between them.

In [ ]:
from onnxruntime.quantization import quantize_dynamic, QuantType
quantize_dynamic('runs/swu-detect/weights/best.onnx',
                 'runs/swu-detect/weights/best-int8.onnx',
                 weight_type=QuantType.QUInt8)
!ls -lh runs/swu-detect/weights/*.onnx

## 9. Download

Save the result as `SWUCardScanner/web/public/models/swu-detect.onnx` (and `swu-detect-int8.onnx`), then start the web app.

In [ ]:
import shutil, os
from google.colab import files

shutil.copy('runs/swu-detect/weights/best.onnx', '/content/swu-detect.onnx')
files.download('/content/swu-detect.onnx')

if os.path.exists('runs/swu-detect/weights/best-int8.onnx'):
    shutil.copy('runs/swu-detect/weights/best-int8.onnx', '/content/swu-detect-int8.onnx')
    files.download('/content/swu-detect-int8.onnx')

# Also keep the .pt - you will want it to fine-tune rather than restart from scratch.
shutil.copy('runs/swu-detect/weights/best.pt', '/content/swu-detect.pt')
files.download('/content/swu-detect.pt')